# Frozen Lake 128x128
The strategy used on the previous notebook does not work on this grid size. I let the model train for 4 hours on aprox 400 epochs, but still, no succes. Therefore a new strategy will be used, still without assuming the gift is in the bottom right to keep the Agent flexible for maps where the gift wouldn't be in the bottom right.

The new strategy will be as follows:
- initialize a q_table with 0s, a 0 indicates an observation the agent hasn't seen yet.
- if the agent falls into a hole, the reward of -2 is given.
- if the agent remains in the same observation, the reward of -1 is given
- If the agent successfully moves without falling into a hole, the reward of 1 is gifted. (this marks the observation as "seen")
- if the agent obtains the gift, credit is assigned for all the steps leading to this step.
- when choosing an action, exploring a new observation is prioritized. (if the observation looks like [-2, -1, 0, 1], it will choose the 0 over the 1, because 0 isn't explored yet and could give out a better reward than the 1)
- We look into the next state to try to predict what state the agent will end up in

imports

In [44]:
import gymnasium as gym
import sys

print(f"python: {sys.version[:7]}")
print(f"gym: {gym.__version__}")

python: 3.10.18
gym: 1.2.0


init env

In [45]:
from gymnasium.envs.toy_text.frozen_lake import generate_random_map
import gymnasium as gym

grid_size = 128

map = generate_random_map(size=grid_size)

env = gym.make(
    'FrozenLake-v1',
    render_mode="rgb_array",
    map_name=f"{grid_size}x{grid_size}", 
    is_slippery=False,
    desc=map
)

initialize q_table

In [46]:
n_actions = env.action_space.n
n_obs = env.observation_space.n
q_table = [[0 for _ in range(n_actions)] for _ in range(n_obs)]

print(n_actions * n_obs)

65536


start learning

In [48]:
import gymnasium as gym
import time
import random

def get_opposite_direction_action(action):
    opposite_actions = {0: 2, 1: 3, 2: 0, 3: 1}
    return opposite_actions[action]

env = gym.make(
    'FrozenLake-v1',
    # render_mode="human",
    render_mode="rgb_array",
    map_name=f"{grid_size}x{grid_size}", 
    is_slippery=False,
    desc=map
)

# parameters
hard_break = False
episode_amount = 10000
max_steps = 25000
total_new_states_learned = 0

epsilon_decay = 1 # dont decay

for episode in range(episode_amount):
    if hard_break:
        break

    # prep episode
    observation, _ = env.reset()
    action_history = []
    observation_history = []
    previous_reward = 0
    credit_decay = 0.999
    new_states_learned = 0
    
    for step in range(max_steps):

        # Choose Action
        possible_actions = q_table[observation].copy()
        if max(possible_actions) > 1: # go to finish is possible
            max_value = max(possible_actions)
            action = possible_actions.index(max_value)

        else: # if path to finish is not clear yet, find it
            try: # try to force new state
                max_value = max(possible_actions)
                if max_value <= 1: # don't force new state if there is a solution leading to the finish
                    action = possible_actions.index(0)
                new_states_learned += 1
            except: # otherwise
                if action_history: # lock opposite direction - prevents moving back
                    last_action = action_history[-1] # get last action
                    possible_actions[get_opposite_direction_action(last_action)] = -0.5 # only move back if the other possible actions are < than -.5
                max_value = max(possible_actions)
                best_actions = [i for i, v in enumerate(possible_actions) if v == max_value] # get each action with the max value
                action = random.choice(best_actions) # greedy option (between the best choices if there are multiple)

        # Environment Step
        new_observation, reward, terminated, truncated, info = env.step(action)
        observation_history.append(observation)
        action_history.append(action)

        # Reward Shaping
        if reward == 1: # Gift Reached +(1+m*c)
            print("Gift Reached!")
            mod_reward = reward
            for observation, action in zip(reversed(observation_history), reversed(action_history)):
                if q_table[observation][action] < reward + mod_reward: # only overwrite if its smaller, so we dont get lost in infinite loops
                    q_table[observation][action] = reward + mod_reward
                    mod_reward = round(mod_reward * credit_decay, ndigits=8)
            hard_break = True
            break
        elif terminated: # Dead -2
            # print("Dead")
            reward = -2
            q_table[observation][action] = reward
        elif observation == new_observation: # Stuck -1
            # print("Stuck")
            reward = -1
            q_table[observation][action] = reward
        elif reward == 0: # Explored +1
            reward = 1
            if q_table[observation][action] < reward: # only overwrite iff the existing reward is smaller
                q_table[observation][action] = reward


        observation = new_observation
        previous_reward = reward

        step+=1

        if terminated:
            break
    print(f'episode: {episode}, steps: {step}, learned: {new_states_learned}')
    total_new_states_learned += new_states_learned
print(f'state learned: {total_new_states_learned}')


episode: 0, steps: 3339, learned: 2
episode: 1, steps: 19745, learned: 1
episode: 2, steps: 11255, learned: 5
episode: 3, steps: 25000, learned: 0
episode: 4, steps: 25000, learned: 0
episode: 5, steps: 12803, learned: 3
episode: 6, steps: 9743, learned: 3
episode: 7, steps: 6052, learned: 1
episode: 8, steps: 8783, learned: 4
episode: 9, steps: 18903, learned: 2
episode: 10, steps: 9617, learned: 4
episode: 11, steps: 8101, learned: 2
episode: 12, steps: 19246, learned: 7
episode: 13, steps: 9285, learned: 1
episode: 14, steps: 10738, learned: 8
episode: 15, steps: 18383, learned: 4
episode: 16, steps: 25000, learned: 0
episode: 17, steps: 8594, learned: 1
episode: 18, steps: 7108, learned: 7
episode: 19, steps: 20097, learned: 2
episode: 20, steps: 25000, learned: 0
episode: 21, steps: 8191, learned: 4
episode: 22, steps: 25000, learned: 0
episode: 23, steps: 17922, learned: 1
episode: 24, steps: 22779, learned: 15
episode: 25, steps: 25000, learned: 0
episode: 26, steps: 9417, learn

q_table after a lot of training

In [41]:
count_m1 = sum(row.count(-1) for row in q_table)
count_0 = sum(row.count(0) for row in q_table)
count_1 = sum(row.count(1) for row in q_table)
count_m1, count_0, count_1

(180, 3601, 8851)

q_table after obtaining the gift

In [49]:
count_m1 = sum(row.count(-1) for row in q_table)
count_0 = sum(row.count(0) for row in q_table)
count_1 = sum(row.count(1) for row in q_table)
count_m1, count_0, count_1

(407, 13723, 33632)

test the agent

In [50]:
import gymnasium as gym
import time
import random

def get_opposite_direction_action(action):
    opposite_actions = {0: 2, 1: 3, 2: 0, 3: 1}
    return opposite_actions[action]

env = gym.make(
    'FrozenLake-v1',
    render_mode="human",
    # render_mode="rgb_array",
    map_name=f"{grid_size}x{grid_size}", 
    is_slippery=False,
    desc=map
)

# parameters
hard_break = False
epochs = 1
episode_amount = 1
max_steps = 500

for epoch in range(epochs):
    if hard_break:
        break

    epsilon = 1 # exploration rate
    epsilon_decay = 1 # dont decay
    
    for episode in range(episode_amount):
        if hard_break:
            break

        # prep episode
        observation, _ = env.reset()
        action_history = []
        observation_history = []
        previous_reward = 0
        credit_decay = 0.999
        
        for step in range(max_steps):

            # Choose Action
            possible_actions = q_table[observation].copy()
            if max(possible_actions) > 1: # go to finish is possible
                max_value = max(possible_actions)
                action = possible_actions.index(max_value)
            else: # if path to finish is not clear yet, find it
                try: # try to force new state
                    max_value = max(possible_actions)
                    if max_value <= 1: # don't force new state if there is a solution leading to the finish
                        action = possible_actions.index(0)
                except: # otherwise
                    if action_history: # lock opposite direction - prevents moving back
                        last_action = action_history[-1] # get last action
                        possible_actions[get_opposite_direction_action(last_action)] = float('-inf')
                    max_value = max(possible_actions)
                    best_actions = [i for i, v in enumerate(possible_actions) if v == max_value] # get each action with the max value
                    action = random.choice(best_actions) # greedy option (between the best choices if there are multiple)

            # Environment Step
            new_observation, reward, terminated, truncated, info = env.step(action)
            print(reward)
            print(terminated)
            print(truncated)
            observation_history.append(observation)
            action_history.append(action)

            # Reward Shaping
            if reward == 1: # Gift Reached +(1+m*c)
                print("Gift Reached!")
                mod_reward = reward
                hard_break = True
                break
            elif terminated: # Dead -2
                print("Dead")
                reward = -2
            elif observation == new_observation: # Stuck -1
                print("Stuck")
                reward = -1
            elif reward == 0: # Explored +1
                reward = 1


            observation = new_observation
            epsilon = epsilon * epsilon_decay
            previous_reward = reward

            step+=1

            if terminated:
                break
    print(f"epoch: {epoch + 1} ended")


0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
False
False
0.0
Fals

In [43]:
q_table

[[-1, 1.71307471, 1, -1],
 [1, -2, 1, -1],
 [1, 1, 1, -1],
 [1, 1, 1, -1],
 [1, 1, 1, -1],
 [1, 1, 1, -1],
 [1, 1, 1, -1],
 [1, 1, -2, -1],
 [0, 0, 0, 0],
 [-2, 1, -2, -1],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [-2, 1, 1, -1],
 [1, 1, -2, -1],
 [0, 0, 0, 0],
 [-2, 1, 1, -1],
 [1, 1, 1, -1],
 [1, 1, 1.37512717, -1],
 [1, 1, 1.37550267, -1],
 [1, 1.37587855, -2, -1],
 [0, 0, 0, 0],
 [-2, 1, 1.37700844, -1],
 [1, 1.37738583, -2, -1],
 [0, 0, 0, 0],
 [-2, 1.38655729, 1, -1],
 [1.38617073, 1, -2, -1],
 [0, 0, 0, 0],
 [-2, 1, -2, -1],
 [0, 0, 0, 0],
 [-2, 1, 1, -1],
 [1, 1, 1, -1],
 [1, 1, 1, -1],
 [1, 1, 1, -1],
 [1, -2, 1, -1],
 [1, -2, -2, -1],
 [0, 0, 0, 0],
 [-2, -2, 1, -1],
 [1, 1, -2, -1],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [-2, -2, 1, -1],
 [1, 1, 1, -1],
 [1, 1, 1, -1],
 [1, -2, 1, -1],
 [1, -2, -2, -1],
 [0, 0, 0, 0],
 [-2, 1, 1, -1],
 [1, 1, 1, -1],
 [1, 1, 1, -1],
 [1, 1, -2, -1],
 [0, 0, 0, 0],
 [0, 0, 0, 0],
 [-2, 1, 1, -1],
 [1, 1, 1, -1],
 [1, -2, -2, -1],
 [0, 0, 0, 0],


In [30]:
q_table

[[-1, 1, 1.9821521899999999, -1],
 [1, 1, 1, -1],
 [1, 1, 1, -1],
 [1, 1.996006, -1, -1],
 [-1, -2, 1, 1],
 [1, 1, 1, 1],
 [1, 1, 1.99202795, 1],
 [1.99302097, 1.9970029999999999, 0, 0],
 [0, 0, 0, 0],
 [-2, 1, 1, 1],
 [1, -2, 1.999, 0],
 [1.998001, 2.0, 0, 0],
 [-1, -1, 1, -2],
 [1, -1, -2, 1],
 [0, 0, 0, 0],
 [0, 0, 0, 0]]

Succes!